In [1]:
import os
import json
import re
import random
import unicodedata
import numpy as np
import pandas as pd
from collections import defaultdict
from difflib import SequenceMatcher
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import spacy

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset, DatasetDict

# ==========================================
# REPRODUCIBILITY & ENV
# ==========================================
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

BASE_DIR = r"D:\student1402\negar\final_research"
WORK_DIR = os.path.join(BASE_DIR, "models", "ner_abstract_optimized_clean")
os.makedirs(WORK_DIR, exist_ok=True)

MODEL_NAME = r"C:\Users\UMZ\.cache\huggingface\hub\models--microsoft--BiomedNLP-PubMedBERT-base-uncased-abstract\snapshots\d673b8835373c6fa116d6d8006b33d48734e305d"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

try:
    nlp = spacy.load("en_ner_bionlp13cg_md")
    print("Loaded BioNLP SciSpaCy model for background entity generation.")
except:
    nlp = spacy.load("en_core_sci_sm")
    print("Loaded fallback SciSpaCy model.")

c:\Users\UMZ\anaconda3\envs\pubmedbert\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Loaded BioNLP SciSpaCy model for background entity generation.


c:\Users\UMZ\anaconda3\envs\pubmedbert\lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [2]:
# ==========================================
# 1a. Load TSV and build pubmed_id → abstract map
# ==========================================
TSV_PATH = os.path.join(BASE_DIR, "data", "lotus_with_title_abstract.tsv")

print("Loading TSV...")
tsv_df = pd.read_csv(TSV_PATH, sep="\t", dtype=str, low_memory=False)
print(f"TSV rows loaded: {len(tsv_df):,}  |  columns: {list(tsv_df.columns)}")

# Build lookup: pubmed_id (str) → abstract text
# Drop rows where abstract is missing
tsv_df = tsv_df.dropna(subset=["reference_pubmed_id", "abstract"])
tsv_df["reference_pubmed_id"] = tsv_df["reference_pubmed_id"].str.strip()
tsv_df["abstract"] = tsv_df["abstract"].str.strip()

pmid_to_abstract = (
    tsv_df[["reference_pubmed_id", "abstract"]]
    .drop_duplicates(subset="reference_pubmed_id")
    .set_index("reference_pubmed_id")["abstract"]
    .to_dict()
)
print(f"Unique PubMed IDs with abstracts in TSV: {len(pmid_to_abstract):,}")



Loading TSV...
TSV rows loaded: 122,733  |  columns: ['structure_wikidata', 'structure_cid', 'structure_nameTraditional', 'organism_wikidata', 'organism_name', 'organism_taxonomy_02kingdom', 'reference_wikidata', 'reference_doi', 'reference_pubmed_id', 'title', 'abstract']
Unique PubMed IDs with abstracts in TSV: 28,864


In [3]:
print("\nLoading lotus_seed_dataset.json...")
with open(os.path.join(BASE_DIR, "data", "lotus_seed_train_clean.json"), encoding="utf-8") as f:
    raw_data = json.load(f)
random.shuffle(raw_data)
print(f"Total records loaded: {len(raw_data):,}")


Loading lotus_seed_dataset.json...
Total records loaded: 85,461


In [4]:
LABEL2ID = {"O": 0, "B-ORG": 1, "I-ORG": 2, "B-CHEM": 3, "I-CHEM": 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# Strictly single-word families to bridge the ontology gap
BROAD_FAMILIES = {
    "alkaloid", "alkaloids", "polyketide", "polyketides", "terpene", "terpenes",
    "terpenoid", "terpenoids", "flavonoid", "flavonoids", "peptide", "peptides",
    "lipopeptide", "lipopeptides", "saponin", "saponins", "coumarin", "coumarins",
    "quinone", "quinones", "steroid", "steroids", "macrolide", "macrolides",
    "glycoside", "glycosides", "sesquiterpene", "sesquiterpenes", "diterpene",
    "diterpenes", "triterpene", "triterpenes", "xanthone", "xanthones"
}

def clean_word(word):
    return re.sub(r'[^\w\s]', '', word).lower()

def create_global_bio_tags(sentence, org_set, chem_set):
    """
    Labels EVERY known entity in the sentence to solve the Partial Labeling problem.
    Longest-match-first so 'Vitamin C' is preferred over 'Vitamin'.
    """
    words = sentence.split()
    labels = ["O"] * len(words)

    def label_spans(entity_set, entity_type):
        sorted_ents = sorted(list(entity_set), key=lambda x: len(x.split()), reverse=True)
        for ent in sorted_ents:
            if not ent:
                continue
            ent_words = ent.split()
            ent_len = len(ent_words)
            for i in range(len(words) - ent_len + 1):
                window = [clean_word(w) for w in words[i:i + ent_len]]
                target = [clean_word(w) for w in ent_words]
                if window == target and labels[i] == "O":
                    labels[i] = f"B-{entity_type}"
                    for j in range(1, ent_len):
                        labels[i + j] = f"I-{entity_type}"

    label_spans(org_set, "ORG")
    label_spans(chem_set, "CHEM")

    # Generic broad-family tags
    for i, w in enumerate(words):
        if labels[i] == "O" and clean_word(w) in BROAD_FAMILIES:
            labels[i] = "B-CHEM"

    return words, labels


def split_abstract_into_sentences(abstract):
    """
    Simple sentence splitter that preserves abbreviations reasonably well.
    Returns a list of non-empty sentence strings.
    """
    # Split on '. ' but not on known abbreviations like 'et al. ' or single capitals
    raw = re.split(r'(?<=[a-z0-9\)])\.\s+', abstract)
    sentences = [s.strip() for s in raw if s.strip()]
    return sentences

In [5]:


from collections import defaultdict

# Group by pubmed_id to collect ALL entity spans per abstract
print("Grouping records by pubmed_id...")
pmid_groups = defaultdict(lambda: {"abstract": "", "org_spans": set(), "chem_spans": set()})

for record in raw_data:
    pmid = str(record.get("pubmed_id", "")).strip()
    abstract  = record.get("abstract", "").strip()
    org_span  = record.get("organism_span", "").strip()
    chem_span = record.get("chemical_span", "").strip()

    if not abstract:
        continue
    if not pmid_groups[pmid]["abstract"]:
        pmid_groups[pmid]["abstract"] = abstract  # store once per paper

    if org_span:  pmid_groups[pmid]["org_spans"].add(org_span)
    if chem_span: pmid_groups[pmid]["chem_spans"].add(chem_span)

print(f"Unique abstracts: {len(pmid_groups):,}")

# Build BIO corpus sentence-by-sentence within each abstract
bio_corpus = []
seen = set()  # (pmid, sentence) pairs

for pmid, group in tqdm(pmid_groups.items(), desc="Building Abstract BIO tags"):
    abstract  = group["abstract"]
    all_orgs  = group["org_spans"]
    all_chems = group["chem_spans"]

    # Run SciSpaCy once per abstract (not per sentence)
    try:
        doc = nlp(abstract[:5000])
        all_orgs  |= {e.text for e in doc.ents if "ORGANISM" in e.label_.upper()}
        all_chems |= {e.text for e in doc.ents if "CHEM"     in e.label_.upper()}
    except Exception:
        pass

    # Split abstract into sentences and tag each one
    sentences = re.split(r'(?<=[a-z0-9\)])\.\s+', abstract)
    for sent in sentences:
        sent = sent.strip()
        if not sent or (pmid, sent) in seen:
            continue
        seen.add((pmid, sent))

        words, labels = create_global_bio_tags(sent, all_orgs, all_chems)
        if "B-ORG" in labels or "B-CHEM" in labels:
            bio_corpus.append({"words": words, "ner_tags": labels})

print(f"Silver-Standard BIO samples ready: {len(bio_corpus):,}")
train_data, val_data = train_test_split(bio_corpus, test_size=0.10, random_state=42)


Grouping records by pubmed_id...
Unique abstracts: 24,614


Building Abstract BIO tags: 100%|██████████| 24614/24614 [06:53<00:00, 59.50it/s]

Silver-Standard BIO samples ready: 88,969


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["words"],
        truncation=True,
        max_length=512,
        is_split_into_words=True
    )
    all_label_ids = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev_word_id:
                label_ids.append(LABEL2ID[word_labels[wid]])
            else:
                label_ids.append(-100)
            prev_word_id = wid
        all_label_ids.append(label_ids)
    tokenized["labels"] = all_label_ids
    return tokenized


raw_ds = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data),
})

tokenized_ds = raw_ds.map(
    tokenize_and_align_labels, batched=True, remove_columns=["words", "ner_tags"]
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

training_args = TrainingArguments(
    output_dir=WORK_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer)
)

print("\nStarting Abstract-Level NER Training...")
trainer.train()
trainer.save_model(WORK_DIR)
tokenizer.save_pretrained(WORK_DIR)
print("✅ NER Training Complete!")

Map: 100%|██████████| 8897/8897 [00:00<00:00, 12707.06 examples/s]
Some weights of BertForTokenClassification were not initialized from the model checkpoint at C:\Users\UMZ\.cache\huggingface\hub\models--microsoft--BiomedNLP-PubMedBERT-base-uncased-abstract\snapshots\d673b8835373c6fa116d6d8006b33d48734e305d and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\UMZ\anaconda3\envs\pubmedbert\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
c:\Users\UMZ\anaconda3\envs\pubmedbert\lib\site-packages\accelerate\accelerator.py:477: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)



Starting Abstract-Level NER Training...


  3%|▎         | 503/15015 [00:26<11:49, 20.47it/s]

{'loss': 0.2436, 'grad_norm': 2.0174155235290527, 'learning_rate': 2.900899100899101e-05, 'epoch': 0.1}


  7%|▋         | 1002/15015 [00:52<11:38, 20.07it/s]

...

{'loss': 0.0945, 'grad_norm': 2.6822705268859863, 'learning_rate': 4.595404595404596e-08, 'epoch': 3.0}


                                                     
100%|██████████| 15015/15015 [13:17<00:00, 19.30it/s]

{'eval_loss': 0.13824772834777832, 'eval_runtime': 5.3628, 'eval_samples_per_second': 1659.015, 'eval_steps_per_second': 52.025, 'epoch': 3.0}


100%|██████████| 15015/15015 [13:18<00:00, 18.81it/s]


{'train_runtime': 798.3546, 'train_samples_per_second': 300.889, 'train_steps_per_second': 18.807, 'train_loss': 0.1273467797896404, 'epoch': 3.0}
✅ NER Training Complete!
